In [66]:
import sympy as sp
import numpy as np
import scipy as sci

In [67]:
# define independent variables 
x,y = sp.symbols('x, y', real=True)
xv = sp.Matrix([x,y])

# define the rhs of the governing equation
w = sp.symbols("omega")
u1,u2 = sp.symbols("u_x, u_y", real=True)
u1 = sp.Function("u_x")(x, y)
u2 = sp.Function("u_y")(x, y)
u = sp.Matrix([u1,u2])
grad_w = sp.Matrix([sp.Derivative(w,x), sp.Derivative(w,y)])
FU = -u.dot(grad_w)
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [68]:
# define q(t)
N = 2 #number of vortexes

q = sp.Matrix()

A = sp.symbols("A", real=True)
L = sp.symbols("L", real=True, positive=True)

# xc= sp.symbols("x_c", real=True)
# yc= sp.symbols("y_c", real=True)

xc= sp.Matrix()
yc= sp.Matrix()
# r = sp.Matrix()

for i in range(N):
    xc = sp.Matrix([xc, sp.symbols("x_c_"+str(i+1), real=True)])
    yc = sp.Matrix([yc, sp.symbols("y_c_"+str(i+1), real=True)])
    # r = sp.Matrix([r, sp.symbols("r_"+str(i+1), real=True, positive=True)])
    # r = sp.Matrix([r, sp.Function("r_"+str(i+1))(x, y, xc[i], yc[i])])

q = sp.Matrix([A, L, xc, yc])
# qr = sp.Matrix([A, L, r])

q

Matrix([
[    A],
[    L],
[x_c_1],
[x_c_2],
[y_c_1],
[y_c_2]])

In [69]:
# define the ansatz u_hat(x; q)
ansatz_gamma = 0
ansatz_gamma = A*sp.exp(-((x-xc[0])**2+(y-yc[0])**2)/L**2) + A*sp.exp(-((x-xc[1])**2+(y-yc[1])**2)/L**2)

ansatz_gamma

A*exp((-(x - x_c_1)**2 - (y - y_c_1)**2)/L**2) + A*exp((-(x - x_c_2)**2 - (y - y_c_2)**2)/L**2)

In [70]:
ansatz_u = sp.Matrix([
    sp.Derivative(ansatz_gamma,y).doit().simplify(),
    -sp.Derivative(ansatz_gamma, x).doit().simplify()
])

ansatz_u.simplify()
ansatz_u

Matrix([
[-2*A*((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2],
[ 2*A*((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2]])

In [71]:
ansatz = (- sp.Derivative(ansatz_gamma, x, 2) - sp.Derivative(ansatz_gamma, y, 2)).doit()
# ansatz = (-sp.Derivative(ansatz_u[0], y) ).doit() + sp.Derivative(ansatz_u[1], x).doit()
ansatz.simplify()

4*A*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4

In [72]:
# compute partial derivatives du/dqi
dwdq = ansatz.diff(q)

dwdq.simplify()

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         4*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4],
[-8*A*exp(-(x - x_c_1)**2/L**2 - (y - y_c_1)**2/L**2)/L**3 - 8*A*exp(-(x - x_c_2)**2/L**2 - (y 

In [73]:
dwdq[0].simplify()

4*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4

In [74]:
# data
a = 1
l = 1
xc1 = 1
yc1 = 1
xc2 = -1
yc2 = -1

def sub_data(f):
    return f.subs(A,a).subs(L,l).subs(xc[0],xc1).subs(yc[0],yc1).subs(xc[1],xc2).subs(yc[1],yc2)

In [75]:
import scipy.integrate as scipy_integrate

def inner_prod_H(f, g, vars=(x, y), BOUND = 30.*l):
    """
    Computes numerical 2D integral of f * g over R^2 for SymPy expressions.
    """
    # Combine expressions and convert directly to a fast numeric function
    integrand_expr = (sub_data(f*g))

    integrand_func = sp.lambdify(vars, integrand_expr, "numpy")

    def integrand(y_val, x_val):
        return integrand_func(x_val, y_val)

    bounds = [[-BOUND, BOUND], [-BOUND, BOUND]]

    # # To integrate x first, then y:
    result, error = scipy_integrate.nquad(
        integrand, bounds            
    )
    
    return result

In [76]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    M[i, i] = inner_prod_H(dwdq[i], dwdq[i])
    print(M[i, i])
    for j in range(i+1, n):
        M[i, j] = inner_prod_H(dwdq[i], dwdq[j])
        M[j,i] = M[i,j]
        print(M[i, j])


25.5930634413449
-27.4343522918661
0.460322212625848
-0.460322212625848
0.460322212624970
-0.460322212624971
95.0070983633220
2.76193327577365
-2.76193327577365
2.76193327577475
-2.76193327577475
37.6991118430700
1.61112774419970
-3.13991901696912e-10
1.84128885051426
37.6991118430700
1.84128885051426
-3.13991901696912e-10
37.6991118430620
1.61112774421155
37.6991118430620


In [77]:
M

Matrix([
[  25.5930634413449, -27.4343522918661,     0.460322212625848,    -0.460322212625848,      0.46032221262497,    -0.460322212624971],
[ -27.4343522918661,   95.007098363322,      2.76193327577365,     -2.76193327577365,      2.76193327577475,     -2.76193327577475],
[ 0.460322212625848,  2.76193327577365,        37.69911184307,       1.6111277441997, -3.13991901696912e-10,      1.84128885051426],
[-0.460322212625848, -2.76193327577365,       1.6111277441997,        37.69911184307,      1.84128885051426, -3.13991901696912e-10],
[  0.46032221262497,  2.76193327577475, -3.13991901696912e-10,      1.84128885051426,       37.699111843062,      1.61112774421155],
[-0.460322212624971, -2.76193327577475,      1.84128885051426, -3.13991901696912e-10,      1.61112774421155,       37.699111843062]])

In [78]:
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [79]:
# compute rhs from the ansatz
Fua = FU.subs(u1, ansatz_u[0]).subs(u2, ansatz_u[1]).subs(w, ansatz).doit()
Fua.simplify()

16*A**2*(((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))*(2*L**2*((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) + (x - x_c_1)**2*(-y + y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)**2*(-y + y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + (-y + y_c_1)**3*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-y + y_c_2)**3*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) - ((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))*(2*L**2*((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) + (-x + x_c_1)**3*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-x + x_c_1)*(y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-x + x_c_2)**3*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + (-x + x_c_2)*(y - y_c

In [80]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(dwdq[i], Fua)
    print(f[i])

f

1.17698906176855e-12
7.38853422888042e-12
-0.639155188734293
0.639155188734293
0.639155188722568
-0.639155188722568


Matrix([
[1.17698906176855e-12],
[7.38853422888042e-12],
[  -0.639155188734293],
[   0.639155188734293],
[   0.639155188722568],
[  -0.639155188722568]])

In [81]:
q_dot = M.inv()*f

q_dot

Matrix([
[2.40541527668503e-13],
[1.68127664568196e-13],
[ -0.0168512375542549],
[  0.0168512375542549],
[  0.0168512375538886],
[ -0.0168512375538886]])